# HINN Colab Training Pipeline
Full end-to-end: environment setup, HINN training, zero-shot LOKO validation, and NSGA-II Pareto search.

**Note:** Visualization is decoupled. Run training and MOO here, then pull results to your local machine for figure generation.

In [ ]:
# Cell 1: Install uv and clone repo
!pip install -q uv
!git clone https://github.com/sattary/2601_chip_paper.git
%cd 2601_chip_paper

In [ ]:
# Cell 2: Pull latest code and install dependencies
!git pull
!uv sync

In [ ]:
# Cell 3: Monotonicity ground truth audit (Paper Section 4.1)
!uv run python cli.py analysis monotonicity --max-groups 5000

In [ ]:
# Cell 4: Train HINN (Standard mode - 500 epochs)
!uv run python cli.py train train --epochs 500 --batch-size 1024 --seed 42

## Zero-Shot Hardware Generalization (LOKO)

In [ ]:
# Cell 5: LOKO Validation (Hold out major kernel)
!uv run python cli.py train train --epochs 500 --loko stencil2d --seed 42

In [ ]:
# Cell 6: Train baselines
!uv run python cli.py train baselines --epochs 500 --seed 42

In [ ]:
# Cell 7: Multi-Objective Optimization (Continuous Frontier Search)
!uv run python cli.py moo run --pop 100 --gen 200

## Push Results to GitHub
Requires `GITHUB_PAT` to be set in Colab Secrets.

In [ ]:
# Cell 8: Authenticated Push
import os
from google.colab import userdata

try:
    pat = userdata.get('GITHUB_PAT')
    repo_url = f"https://{pat}@github.com/sattary/2601_chip_paper.git"
    
    !git config --global user.email "thesattary@gmail.com"
    !git config --global user.name "sat"
    !git add results/  # Syncs all models and Pareto CSVs
    !git commit -m "HINN compute results sync"
    !git remote set-url origin {repo_url}
    !git push origin master
    print("Results successfully pushed to GitHub!")
except Exception as e:
    print(f"Sync failed: {e}")